# FinSight AI on Colab (free)
Runtime → **Change runtime type → T4 GPU**, then run the cells top to bottom.

* Embeddings, reranker **and the LLM** (Qwen2.5-3B-Instruct, 4-bit) run on the Colab GPU — no API key, no cost.
* To pick up new code you pushed from your laptop, just re-run cell 1.

## 1. Get the code

In [ ]:
import os, subprocess
REPO = "https://github.com/yashbhangdia/Rag-citation-app"   # private repo? use a token from Colab Secrets (below)
try:
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN")           # optional: Colab → 🔑 Secrets
    if tok: REPO = REPO.replace("https://", f"https://{tok}@")
except Exception:
    pass
if os.path.isdir("/content/app/.git"):
    subprocess.run(["git", "-C", "/content/app", "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO, "/content/app"], check=True)
%cd /content/app

In [ ]:
!pip install -q -r requirements-colab.txt

## 2. Storage
Mount Google Drive so the vector index and PDFs survive session resets (otherwise they live on the wiped VM disk).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DATA = "/content/drive/MyDrive/finsight"
os.makedirs(f"{DATA}/documents", exist_ok=True)
os.environ["CHROMA_PERSIST_DIR"] = f"{DATA}/chroma_db"
os.environ["DOCUMENTS_DIR"] = f"{DATA}/documents"
os.environ["LLM_BACKEND"] = "transformers"
print("index:", os.environ["CHROMA_PERSIST_DIR"])

## 3. Documents
Either drop your own PDFs into `MyDrive/finsight/documents/` (naming like `TCS_Annual_Report_FY2024-25.pdf` gives company / year metadata), or download the sample filings:

In [ ]:
# Downloads public filings into the repo layout, then copies them to Drive. Skip if you uploaded your own.
!python scripts/download_documents.py --all
!cp -rn data/documents/* "$DOCUMENTS_DIR"/ 2>/dev/null || true
!ls -R "$DOCUMENTS_DIR" | head -30

## 4. Build the index (skips nothing — add `--reset` to rebuild; re-running upserts by chunk id)

In [ ]:
!python scripts/ingest.py --dir "$DOCUMENTS_DIR"

## 5. Load the models and launch the chat (first run downloads ~6 GB of weights)

In [ ]:
import sys; sys.path.insert(0, "/content/app")
from src.generation.rag_pipeline import RAGPipeline
from frontend.gradio_app import build_demo
pipeline = RAGPipeline()          # TransformersLLM (4-bit on the T4) + hybrid retrieval + reranker
print("LLM:", pipeline.llm.model)
build_demo(pipeline).queue().launch(share=True, debug=False)

### Optional: measure quality
```
!python scripts/eval.py --generate
```